In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from pathlib import Path

unsupp_dir = "code/training/encoder_training/scibert/runs/4_unsupported_short_scibert/checkpoint-21"
tokenizer = AutoTokenizer.from_pretrained(unsupp_dir, local_files_only=True, model_max_length=256)
model = AutoModelForTokenClassification.from_pretrained(unsupp_dir, local_files_only=True)

In [2]:
text = """ A common issue occurs that can potentially impact the validity of results is filtering the set of systems to be evaluated via automatic metric scores. Since metric scores are known to be a poor substitute for human assessment, this only results in the pos-sibility that the best system according to human judges is inadvertently filtered out at this stage. For example, ConvAI2 (Dinan et al., 2019) ranked models firstly using automatic metrics before top models according to metric scores were assessed by crowd-sourced workers on Mechanical Turk, while similarly in the sixth Dialog System Technology Challenge (DSTC6) systems were filtered according to metric scores prior to human evaluation.

In terms of the live evaluation, competitions such as Convai2 report such evaluations as highly challenging, with many of the resulting dialogues reported to be senseless, offensive, or simply not in line with instructions and ultimately live evaluation results have been discarded.

Despite challenges, competitions that operate in the public domain, making data and evaluation techniques available to researchers (such as ourselves) should be applauded for such efforts.

On the other hand, competitions that (for one reason or another) do not release data and evaluation techniques into the public domain have reported relative success in terms of human evaluation. However until such methods can be accessed and independently verified through replication studies, they will unfortunately have little impact . The first Amazon Alexa Socialbot Grand Challenge required human assessors to score how coherent and engaging conversations were on a 1-5 rating scale by two distinct groups: volunteer Amazon employees (experts), and general Alexa users (crowds) (Ram et al., 2018), are reported to achieve a correlation of overall scores for the two types of human assessors at 0.93. The absolute average rating across all chatbots was reported to be 20% lower for experts compared to general users. In an additional effort to evaluate models, conversational user experience, coherence, engagement, domain coverage, topical diversity, and conversational depth were assessed (1-5 scale), with combined scores reported to correlate with those of general users at r = 0.66. In addition to methods and data not being publicly available, correlations are difficult to interpret since no detail is provided about the number of judgments on which the correlation is calculated for example.

In addition to competitions that generally aim to include human evaluation of systems, automatic metrics are often proposed for dialogue evaluation, themselves requiring a human evaluation data set on which to evaluate the proposed metric. How-ever, inappropriate statistics are often applied. For example, Pang et al. (2020) propose a holistic metric to automatically evaluate four distinct aspects of dialogue, and a human evaluation experiment is deployed on Mechanical Turk using a 1-5 rating scale. The mean correlation between human assessors is reported as r = 0.61. However, mean correlations are unfortunately difficult to interpret, since correlation coefficients are not additive , averages calculated in the usual way cannot be assumed to reflect central tendency, and unfortunately, the distribution of correlations is not reported (Alexander, 1990). Mehri and Eskenazi (2020b) propose USR (Un-Supervised and Reference-free), an unsupervised model that predicts the quality of dialog for a range of criteria using various rating scales: understandable (0-1 rating scale), natural (1-3), maintains context (1-3), interesting (1-3), uses knowledge (0-1); overall quality (1-5). Despite human evaluation being carried out by experts inter-annotator agreement levels varied depending on criteria being measured, ranging from as low as 0.298. Additionally, although correlations between human assessments are reported as significant at p < 0.01, despite such statistics often being reported for correlations, they are unfortunately not very meaningful in terms of their impact on correlation interpretation and can be somewhat misleading. Contrary to common expectations, even small effect sizes (low r) can produce very low p-values (strong significance) in such tests. Aiming to achieve a significant correlation is an extremely low bar to reach in terms of consistency, since a low p-value in this case simply rejects the null hypothesis that the correlation is zero.

In addition to the above issues, human evaluation of dialogue systems rarely take into account the fact that differences in performance can occur simply by chance. The method of human evaluation we propose provides a means of applying standard tests for statistical significance to avoid concluding differences that are highly likely to have occurred simply by chance.
"""

In [83]:
import spacy
import torch

def predict_subwords(text: str):
    """
    Returns a list of (subword_token, label) excluding special tokens.
    Tokens are WordPiece-style: 'conv', '##ai', etc.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred_ids = outputs.logits.argmax(dim=-1)[0].tolist()
    input_ids = inputs["input_ids"][0].tolist()

    toks = tokenizer.convert_ids_to_tokens(input_ids)
    labs = [model.config.id2label[int(i)] for i in pred_ids]

    out = []
    for tok, lab in zip(toks, labs):
        if tok in (tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token):
            continue
        out.append((tok, lab))
    return out


def merge_wordpiece_span(tokens):
    """
    Merge a list of WordPiece tokens into a single string.
    Example: ['conv', '##ai'] -> 'convai'
    """
    s = ""
    for t in tokens:
        if t.startswith("##"):
            s += t[2:]
        else:
            # add a space only if this isn't the first token AND the previous merge ended a word
            if s != "":
                # if you prefer no spaces between separate words in a span, remove this block
                s += " "
            s += t
    return s


nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
sentences = [sent.text.strip() for sent in doc.sents]

entities = {}
predictions = []

for sent in sentences:
    pred = predict_subwords(sent)   # list of (subword, label)
    predictions.append(pred)

    i = 0
    while i < len(pred):
        tok, lab = pred[i]

        if lab != "ENTITY":
            i += 1
            continue

        # start of an ENTITY subword span
        start = i
        i += 1
        while i < len(pred) and pred[i][1] == "ENTITY":
            i += 1
        end = i  # exclusive

        span_subwords = [t for t, _ in pred[start:end]]
        
        if (end - start) == 1:
            continue
        span_text = merge_wordpiece_span(span_subwords)

        # Count the whole span as ONE entity
        entities[span_text] = entities.get(span_text, 0) + 1

In [84]:
# to do: find first mentions of "entity" spans in text, but b4 that, check if entity span is a noun 
entities

{'convai': 1,
 'crowd - sourced workers': 1,
 'dialog system technology': 1,
 'amazon alexa socialbot grand': 1,
 'amazon employees': 1,
 'dialogue evaluation': 1,
 'un - supervised': 1,
 'reference - free': 1,
 'unsupervised model': 1,
 'dialogue systems': 1}

In [102]:
import re
from typing import List, Dict, Any, Optional

citation_pattern = re.compile(
    r"""
    (
        \([A-Z][A-Za-z\-]+(?:\s+(?:et\ al\.|&|and)\s+[A-Z][A-Za-z\-]+)?\,?\s+\d{4}[a-z]?\)
        |
        [A-Z][A-Za-z\-]+(?:\s+(?:et\ al\.|&|and)\s+[A-Z][A-Za-z\-]+)?\s+\(\d{4}[a-z]?\)
        |
        \[(\d+([,\-\u2013]\s*\d+)*)\]
    )
    """,
    re.VERBOSE,
)

def citation_after_mention(
    sentence: str,
    span: str,
    *,
    window_chars: int = 200,          # window AFTER the mention
    pre_window_chars: int = 80,       # window BEFORE the mention to allow "(X, 2020), SPAN"
    case_insensitive: bool = True,
) -> Optional[Dict[str, Any]]:
    """
    If span appears in sentence:
      - primary: look AFTER the first occurrence of span for citation (within window_chars)
      - secondary: allow citation BEFORE span if:
          * there's a citation within pre_window_chars before span
          * and there is a comma between the citation and the span within that pre-window
        Example: "(Smith, 2020), Yoopick ..."
    Returns None if span not in sentence.
    """
    if not span:
        return None

    flags = re.IGNORECASE if case_insensitive else 0
    m = re.search(re.escape(span), sentence, flags)
    if not m:
        return None

    span_start, span_end = m.start(), m.end()

    # --- check AFTER span ---
    after = sentence[span_end : span_end + max(0, int(window_chars))]
    cit_after = citation_pattern.search(after)
    if cit_after:
        return {
            "span": span,
            "sentence": sentence,
            "sentence_index": None,
            "mention_start": span_start,
            "mention_end": span_end,
            "citation_found": True,
            "citation_where": "after",
            "citation_match": cit_after.group(0),
        }

    # --- check BEFORE span with comma rule ---
    pre_w = max(0, int(pre_window_chars))
    before_start = max(0, span_start - pre_w)
    before = sentence[before_start:span_start]

    # find the LAST citation in the before-window (closest to span)
    last_cit = None
    for mm in citation_pattern.finditer(before):
        last_cit = mm

    if last_cit:
        # text between citation end and span start (within the before-window)
        between = before[last_cit.end():]
        # require a comma somewhere between citation and span
        if "," in between:
            return {
                "span": span,
                "sentence": sentence,
                "sentence_index": None,
                "mention_start": span_start,
                "mention_end": span_end,
                "citation_found": True,
                "citation_where": "before_with_comma",
                "citation_match": last_cit.group(0),
            }

    # --- no citation found (after or before-with-comma) ---
    return {
        "span": span,
        "sentence": sentence,
        "sentence_index": None,
        "mention_start": span_start,
        "mention_end": span_end,
        "citation_found": False,
        "citation_where": None,
        "citation_match": None,
    }

def find_first_mentions_missing_citation(
    sentences: List[str],
    spans: List[str],
    *,
    window_chars: int = 200,
    pre_window_chars: int = 80,
    case_insensitive: bool = True,
) -> List[Dict[str, Any]]:
    """
    For each span, locate its FIRST mention across sentences (by order),
    then check whether a citation appears after the mention OR before-with-comma.
    Returns ONLY the ones where citation is missing.
    """
    missing = []

    for span in spans:
        first_result = None
        first_sent_idx = None

        for i, sent in enumerate(sentences):
            res = citation_after_mention(
                sent,
                span,
                window_chars=window_chars,
                pre_window_chars=pre_window_chars,
                case_insensitive=case_insensitive,
            )
            if res is not None:
                first_result = res
                first_sent_idx = i
                break

        if first_result is None:
            continue

        first_result["sentence_index"] = first_sent_idx

        if not first_result["citation_found"]:
            missing.append(first_result)

    return missing

# ---- usage ----
spans = list(entities.keys())

problematic = find_first_mentions_missing_citation(
    sentences,
    spans,
    window_chars=200,
    pre_window_chars=70,  # adjust if you want to allow longer "(cite), ... SPAN" gaps
    case_insensitive=True,
)

for r in problematic:
    print("SPAN:", r["span"])
    print("SENT IDX:", r["sentence_index"])
    print("SENT:", r["sentence"])
    print("----")

SPAN: convai
SENT IDX: 2
SENT: For example, ConvAI2 (Dinan et al., 2019) ranked models firstly using automatic metrics before top models according to metric scores were assessed by crowd-sourced workers on Mechanical Turk, while similarly in the sixth Dialog System Technology Challenge (DSTC6) systems were filtered according to metric scores prior to human evaluation.
----
SPAN: dialog system technology
SENT IDX: 2
SENT: For example, ConvAI2 (Dinan et al., 2019) ranked models firstly using automatic metrics before top models according to metric scores were assessed by crowd-sourced workers on Mechanical Turk, while similarly in the sixth Dialog System Technology Challenge (DSTC6) systems were filtered according to metric scores prior to human evaluation.
----
SPAN: amazon alexa socialbot grand
SENT IDX: 7
SENT: The first Amazon Alexa Socialbot Grand Challenge required human assessors to score how coherent and engaging conversations were on a 1-5 rating scale by two distinct groups: vol